# ==============================================================================
# PROYECTO INTEGRACIÓN Y TRANSFORMACIÓN DE DATOS - AVANCE 2
# Título: Análisis De Las Brechas Entre Oferta y Demanda Laboral En el Tolima
# Integrantes: Karen Vanesa Cardona, María Fernanda Rodríguez, Karen Natalia Moreno
# Universidad de Ibagué - Noviembre de 2026
# ==============================================================================

In [17]:
# ------------------------------------------------------------------------------
# CELDA 1: Importación de Librerías
# ------------------------------------------------------------------------------
import pandas as pd
import numpy as np
import io
import os
from google.colab import files

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


In [18]:
# ------------------------------------------------------------------------------
# CELDA 2: Carga de Archivos desde tu Equipo (o Google Drive)
# ------------------------------------------------------------------------------
# Ejecuta esta celda para subir 'COMFATOLIMA.xlsx' y 'dane.xlsx' desde tu PC
print("Por favor, sube los archivos 'COMFATOLIMA.xlsx' y 'dane.xlsx':")
uploaded = files.upload()

path_comfatolima = 'COMFATOLIMA.xlsx'
path_dane = 'dane.xlsx'

Por favor, sube los archivos 'COMFATOLIMA.xlsx' y 'dane.xlsx':


Saving COMFATOLIMA.xlsx to COMFATOLIMA (1).xlsx
Saving dane.xlsx to dane (1).xlsx


In [19]:
# ------------------------------------------------------------------------------
# CELDA 3: Extracción Inicial (Hito 1 - Extracción Técnica)
## ------------------------------------------------------------------------------
print("=== 1. EXTRAENDO DATOS INICIALES ===")
df_comfa_raw = pd.read_excel(path_comfatolima)
df_dane_raw = pd.read_excel(path_dane, sheet_name='Hoja1')
print(f"-> COMFATOLIMA cargado: {df_comfa_raw.shape[0]} filas, {df_comfa_raw.shape[1]} columnas")
print(f"-> DANE cargado: {df_dane_raw.shape[0]} filas, {df_dane_raw.shape[1]} columnas")

=== 1. EXTRAENDO DATOS INICIALES ===
-> COMFATOLIMA cargado: 1000 filas, 20 columnas
-> DANE cargado: 16 filas, 20 columnas


In [20]:
# ------------------------------------------------------------------------------
# CELDA 4: Ejecución de las 5 Transformaciones Esenciales
# ------------------------------------------------------------------------------
print("\n=== 2. APLICANDO TRANSFORMACIONES ===")


=== 2. APLICANDO TRANSFORMACIONES ===


In [21]:
# --- TRANSFORMACIÓN 1: Limpieza de Encoding (Mojibake UTF-8) ---
def corregir_encoding(texto):
    if not isinstance(texto, str):
        return texto
    reemplazos = {
        'Ã': 'Í', 'Ã³': 'ó', 'Ã­': 'í', 'Ã‘': 'Ñ', 'Ã±': 'ñ',
        'Ã¡': 'á', 'Ã©': 'é', 'Ãº': 'ú', 'Â': '', '$hNo Codigo vacante$': ''
    }
    for clave, valor in reemplazos.items():
        texto = texto.replace(clave, valor)
    return texto.strip()

df_comfa_clean = df_comfa_raw.copy()
columnas_texto = ['TITULO_VACANTE', 'DESCRIPCION_VACANTE', 'CARGO', 'SECTOR_ECONOMICO']
for col in columnas_texto:
    df_comfa_clean[col] = df_comfa_clean[col].apply(corregir_encoding)
print("[T1 Completada] Corregida codificación UTF-8 / Mojibake.")

[T1 Completada] Corregida codificación UTF-8 / Mojibake.


In [7]:
# --- TRANSFORMACIÓN 2: Desduplicación y Depuración de Outliers ---
df_comfa_clean = df_comfa_clean.drop_duplicates(subset=['CODIGO_VACANTE'])
# Tratar experiencia > 120 meses (10 años) como valor atípico / nulo
df_comfa_clean.loc[df_comfa_clean['MESES_EXPERIENCIA_CARGO'] > 120, 'MESES_EXPERIENCIA_CARGO'] = np.nan
print("[T2 Completada] Eliminados duplicados y corregidos outliers en meses de experiencia.")

[T2 Completada] Eliminados duplicados y corregidos outliers en meses de experiencia.


In [8]:
# --- TRANSFORMACIÓN 3: Mapeo y Estandarización de Rangos Salariales a Valores Numéricos ---
salario_map = {
    '$1.000.001 - $1.500.000': 1250000,
    '$1.500.001 - $2.000.000': 1750000,
    '$2.000.001 - $3.000.000': 2500000,
    '$3.000.001 - $4.000.000': 3500000,
    '$4.000.001 - $6.000.000': 5000000,
    '$6.000.001 - $8.000.000': 7000000,
    'Mayor de $15.000.001': 15000000,
    '$589.500 - $884.250': 736875,
    '$884.251 - $1.000.000': 942125,
    'Menos del Salario Mínimo': 650000,
    'A Convenir': np.nan
}
df_comfa_clean['SALARIO_ESTIMADO_NUM'] = df_comfa_clean['RANGO_SALARIAL'].map(salario_map)
print("[T3 Completada] Rango salarial convertido a valores cuantitativos continuos.")

[T3 Completada] Rango salarial convertido a valores cuantitativos continuos.


In [9]:
# --- TRANSFORMACIÓN 4: Normalización Unpivot (Melt) de la Fuente DANE ---
df_dane_clean = df_dane_raw.dropna(subset=['Tolima']).copy()
df_dane_clean.columns = ['CONCEPTO'] + [str(int(x)) for x in df_dane_raw.iloc[1, 1:].values]
df_dane_clean = df_dane_clean[df_dane_clean['CONCEPTO'] != 'Concepto']

df_dane_melted = df_dane_clean.melt(
    id_vars=['CONCEPTO'],
    var_name='ANIO',
    value_name='VALOR'
)
df_dane_melted['ANIO'] = df_dane_melted['ANIO'].astype(int)
print("[T4 Completada] Matriz horizontal de DANE convertida a formato tabular Tidy (Unpivot).")

[T4 Completada] Matriz horizontal de DANE convertida a formato tabular Tidy (Unpivot).


In [10]:
# --- TRANSFORMACIÓN 5: Segmentación Geográfica Departamental (Tolima) ---
df_vacantes_tolima = df_comfa_clean[df_comfa_clean['DEPARTAMENTO'].str.upper() == 'TOLIMA'].copy()
print(f"[T5 Completada] Filtradas ofertas pertenecientes al Tolima ({len(df_vacantes_tolima)} ofertas encontradas).")

[T5 Completada] Filtradas ofertas pertenecientes al Tolima (11 ofertas encontradas).


In [11]:
# ------------------------------------------------------------------------------
# CELDA 5: Controles de Calidad Automatizados (Hito 3)
# ------------------------------------------------------------------------------
print("\n=== 3. CONTROLES DE CALIDAD E INTEGRIDAD ===")


=== 3. CONTROLES DE CALIDAD E INTEGRIDAD ===


In [12]:
# Control 1: Unicidad de Claves Primarias
assert df_vacantes_tolima['CODIGO_VACANTE'].is_unique, "Error: Existen vacantes duplicadas."
print("[Control 1 OK] Unicidad de código de vacante verificado.")

# Control 2: Validación de Nulos en Métricas Críticas
assert not df_vacantes_tolima['CANTIDAD_VACANTES'].isnull().any(), "Error: Hay vacantes con plazas nulas."
print("[Control 2 OK] Ausencia de nulos en métricas principales.")

# Control 3: Integridad Geográfica
municipios_validos = ['IBAGUÉ', 'ESPINAL', 'HONDA', 'MELGAR']
assert df_vacantes_tolima['MUNICIPIO'].isin(municipios_validos).all(), "Error: Existen municipios fuera de cobertura."
print("[Control 3 OK] Todos los municipios corresponden al Tolima.")

[Control 1 OK] Unicidad de código de vacante verificado.
[Control 2 OK] Ausencia de nulos en métricas principales.
[Control 3 OK] Todos los municipios corresponden al Tolima.


In [13]:
# ------------------------------------------------------------------------------
# CELDA 6: Generación y Exportación de Datasets Consolidados (Hito 4)
# ------------------------------------------------------------------------------
print("\n=== 4. EXPORTANDO DATASETS CONSOLIDADOS LIMPIOS ===")

df_vacantes_tolima.to_csv('Fact_Vacantes_Tolima_Limpio.csv', index=False)
df_dane_melted.to_csv('Fact_Indicadores_DANE_Tolima_Limpio.csv', index=False)

print("Archivos exportados exitosamente:")
print(" - Fact_Vacantes_Tolima_Limpio.csv")
print(" - Fact_Indicadores_DANE_Tolima_Limpio.csv")


=== 4. EXPORTANDO DATASETS CONSOLIDADOS LIMPIOS ===
Archivos exportados exitosamente:
 - Fact_Vacantes_Tolima_Limpio.csv
 - Fact_Indicadores_DANE_Tolima_Limpio.csv


In [14]:
# Descargar automáticamente a tu computador desde Colab
files.download('Fact_Vacantes_Tolima_Limpio.csv')
files.download('Fact_Indicadores_DANE_Tolima_Limpio.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
# ------------------------------------------------------------------------------
# CELDA 7: Vista Previa de los Resultados
# ------------------------------------------------------------------------------
print("\n=== VISTA PREVIA: VACANTES TOLIMA CONSOLIDADAS ===")
display(df_vacantes_tolima[['CODIGO_VACANTE', 'TITULO_VACANTE', 'MUNICIPIO', 'CANTIDAD_VACANTES', 'NIVEL_ESTUDIOS', 'SALARIO_ESTIMADO_NUM']])

print("\n=== VISTA PREVIA: INDICADORES DANE NORMALIZADOS ===")
display(df_dane_melted.head(10))


=== VISTA PREVIA: VACANTES TOLIMA CONSOLIDADAS ===


,CODIGO_VACANTE,TITULO_VACANTE,MUNICIPIO,CANTIDAD_VACANTES,NIVEL_ESTUDIOS,SALARIO_ESTIMADO_NUM
37,66788522,Lider operativo de despacho - Bodega,ESPINAL,1,Universitarios,2500000.0
228,66782475,Auxiliar Facturador/a - Con experiencia,HONDA,2,Técnico,1750000.0
302,1626551153-1,Auxiliar de campo,ESPINAL,3,No Aplica,1750000.0
321,1626448236-201,AUXILIAR DE BODEGA/ALMACÍ‰N (MELGAR- TOLIMA),MELGAR,1,Técnico,1750000.0
324,66788502,vendedor minimercados,IBAGUÉ,3,Media,1750000.0
430,1626540669-3,AUXILIAR CONTABLE,IBAGUÉ,1,Técnico,1750000.0
584,1626403719-4,AUXILAR RESIDENTE DE INTERVENTORIA (ARQUITECTO),IBAGUÉ,1,Universitarios,1750000.0
591,66781597,Auxiliares de Bodega Almacén o Logística - Iba...,IBAGUÉ,30,Media,2500000.0
621,1626401514-11,TELEMERCADERISTA,IBAGUÉ,5,Técnico,1750000.0
646,66789602,Entrevistador Motorizado - Ibague,IBAGUÉ,10,Básica Secundaria,NaN



=== VISTA PREVIA: INDICADORES DANE NORMALIZADOS ===


,CONCEPTO,ANIO,VALOR
0,% población en edad de trabajar,2007,71.943644
1,Tasa Global de Participación (TGP),2007,65.638333
2,Tasa de Ocupación (TO),2007,57.147105
3,Tasa de Desocupación (TD),2007,12.936386
4,Tasa de Subocupación (TS),2007,12.111840
5,Población total,2007,1307.832000
6,Población en edad de trabajar (PET),2007,940.902000
7,Fuerza de trabajo,2007,617.592387
8,Población ocupada,2007,537.698253
9,Población desocupada,2007,79.894134
